## Chosen corpus: The Romanian Constitution
https://www.presidency.ro/ro/constitutia-romaniei

In [258]:
def load_corpus(path: str) -> str:
    with open(path, encoding="utf-8") as file:
        return file.read()

In [ ]:
def tokenize_corpus(corpus: str, insert_sentence_delimeters: bool = True) -> list[str]:
    tokens = []

    end_of_sentence_punctuation = ".?!"
    punctuation = end_of_sentence_punctuation + ",;:-\"()"
    whitespace = " \t\n\r\t\f"
    delimeters = whitespace + punctuation
    word = ""

    new_corpus = f"<s> {corpus} </s> " if insert_sentence_delimeters else f"{corpus} "

    for character in new_corpus:
        if character in delimeters:
            if len(word) > 0:
                tokens.append(word.lower())
                word = ""
            if character in punctuation:
                tokens.append(character)

        else:
            word += character

        # Sentence Delimitation
        if insert_sentence_delimeters and len(tokens) > 0:
            if (tokens[-1] != "<s>"  and character == "\n") or (tokens[-1] in end_of_sentence_punctuation and character.isupper()):
                tokens.extend(["</s>", "<s>"])
    
    return tokens

In [260]:
from IPython.display import clear_output

def input_ngram_number() -> int:
    while True:
        print("Input an 'N' for the N-gram:\n * N = 1 for unigram\n * N = 2 for bigram\n * N = 3 for trigram")
        n = int(input("N = "))
        if n in [1, 2, 3]:
            return n
        else:
            clear_output()
            print(f"N = {n} is not a valid 'N' for an N-gram. Please try again")

In [261]:
def compute_counts(tokens: list[str], ngram_number: int, break_context_on_sentence_end: bool = True) -> dict[tuple[str], dict[str, int]]:
    counts = {}

    if ngram_number not in [1, 2, 3]:
        return counts

    if len(tokens) < ngram_number:
        return counts
    
    context_length = ngram_number - 1
    context: tuple[str] = tuple()
    for token in tokens:
        if len(context) == context_length:
            if not context in counts:
                counts[context] = {}

            if token in counts[context]:
                counts[context][token] += 1
            else:
                counts[context][token] = 1

        if break_context_on_sentence_end and token == "</s>":
            context = tuple()
        elif len(context) < context_length:
            context = context + tuple([token])
        else:
            context = (context[1:] + tuple([token]))[:context_length]

    return counts
    

In [262]:
def compute_probabilities(counts: dict[tuple[str], dict[str, int]], vocab_length: int, apply_laplace_add_one: bool) -> dict[tuple[str], dict[str, float]]:
    probabilities = {}
    for context, token_counts in counts.items():
        probabilities[context] = {}
        context_count = sum(ngram_count for ngram_count in token_counts.values())
        for token, ngram_count in token_counts.items():
            if apply_laplace_add_one:
                probabilities[context][token] = (ngram_count + 1) / (context_count + vocab_length)
            else:
                probabilities[context][token] = ngram_count / context_count

    return probabilities

In [263]:
def get_ngram_probability(probabilities: dict[tuple[str], dict[str, float]], context: tuple[str], token: str, vocab_length: int, apply_laplace_add_one: bool) -> float:
    if context not in probabilities or token not in probabilities[context]:
        if apply_laplace_add_one:
            return 1 / vocab_length
        else:
            return 0
    else:
        return probabilities[context][token]

In [264]:
def count_distinct_ngrams(counts: dict[tuple[str], dict[str, int]]) -> int:
    return sum(len(ngrams) for ngrams in counts.values())

In [265]:
def most_frequent_ngrams(counts: dict[tuple[str], dict[str, int]], top_k: int) -> list[tuple[tuple[str], int]]:
    return sorted(
        (
            (context + tuple([token]), ngram_count) 
                for context, token_counts in counts.items() 
                    for token, ngram_count in token_counts.items()
        ),
        key=lambda x: x[1],
        reverse=True
    )[: top_k]

In [266]:
def compare_probabilities(counts: dict[tuple[str], dict[str, int]], vocab_length: int, ngram_number: int, test_ngram: tuple[str]):
        probabilities = compute_probabilities(counts, vocab_length, apply_laplace_add_one=False)
        probability = get_ngram_probability(probabilities, test_ngram[:-1], test_ngram[-1], vocab_length, apply_laplace_add_one=False)
        print(f"Conditional probability of {ngram_number}-gram {test_ngram} before smoothing: {probability}")
    
        smoothed_probabilities = compute_probabilities(counts, vocab_length, apply_laplace_add_one=True)
        smoothed_probability = get_ngram_probability(smoothed_probabilities, test_ngram[:-1], test_ngram[-1], vocab_length, apply_laplace_add_one=True)
        print(f"Conditional probability of {ngram_number}-gram {test_ngram} after smoothing: {smoothed_probability}")

In [267]:
from pprint import pprint

def main():
    ngram_number = input_ngram_number()

    print(f"\nN = {ngram_number} \n")

    corpus = load_corpus("romanian_constitution.txt")
    corpus_tokens = tokenize_corpus(corpus)
    vocab = set(corpus_tokens)

    print(f"Number of tokens: {len(corpus_tokens)}")
    print(f"Vocabulary size: {len(vocab)}")

    counts = compute_counts(corpus_tokens, ngram_number)

    distinct_ngram_count = count_distinct_ngrams(counts)
    print(f"Number of distinct {ngram_number}-grams: {distinct_ngram_count}")

    topk_frequent_ngrams = most_frequent_ngrams(counts, 20)
    print(f"\nMost frequent {ngram_number}-grams:")
    pprint(topk_frequent_ngrams)

    print()
    compare_probabilities(counts, len(vocab), ngram_number, topk_frequent_ngrams[0][0])
    
    print()
    compare_probabilities(counts, len(vocab), ngram_number, ('elefantul', 'a', 'mers')[:ngram_number])
    
    

In [268]:
main()

Input an 'N' for the N-gram:
 * N = 1 for unigram
 * N = 2 for bigram
 * N = 3 for trigram

N = 1 

Number of tokens: 16984
Vocabulary size: 2578
Number of distinct 1-grams: 2578

Most frequent 1-grams:
[(('<s>',), 973),
 (('</s>',), 973),
 ((',',), 783),
 (('de',), 679),
 (('.',), 550),
 (('şi',), 533),
 ((')',), 470),
 (('(',), 460),
 (('în',), 421),
 (('a',), 287),
 (('la',), 216),
 (('se',), 182),
 (('articolul',), 181),
 (('sau',), 162),
 (('2',), 144),
 (('1',), 138),
 (('cu',), 111),
 (('este',), 110),
 (('3',), 107),
 (('prin',), 101)]

Conditional probability of 1-gram ('<s>',) before smoothing: 0.057289213377296276
Conditional probability of 1-gram ('<s>',) after smoothing: 0.0497904099785298

Conditional probability of 1-gram ('elefantul',) before smoothing: 0
Conditional probability of 1-gram ('elefantul',) after smoothing: 0.0003878975950349108
